## Анализ рынка недвижимости
### Цель
Изучить, какие признаки и в какой степени влияют на формирование цен на недвижимость.
### Сбор и хранение данных
Для анализа были собраны и сохранены несколько тысяч объявлений о продаже квартир.  
Основным источником на момент сбора данных был сайт cian.ru  
*Собранный материал используется исключительно в образовательных целях в сфере машинного обучения.*  
*Личные данные пользователей с сайта cian.ru не собирались, остальное содержимое базы данных также не публикуется.*  
*Парсер сайта намеренно не оптимизируется, чтобы не усиливать потенциальную угрозу распространения конфиденциальной информации о пользователях cian.ru.*  
База данных SQLite3 (далее БД) состоит из трех основных таблиц:
- **apartments** - содержит основные характеристики самих объектов сделки
- **listings** - содержит характеристики объявлений
- **price_history** - содержит историю изменения цен внутри каждого объявления  
### Ограниченность и специфика данных
- Исследование проводится не на фактических суммах сделок, а на суммах, предоженных продавцами.
- Выборка ограничена относительно небольшим списком городов, в которых размещены объявления, и имеет большой перекос в пользу некоторых из них, что связано как с особенностями работы парсера, так и с величиной самих городов.
- При сборе имеющихся объявлений парсер не работал непрерывно и запускался *время от времени*.
- По состоянию БД на 9.11.2026 временной диапазон публикации объявлений: ***с июля 2019г по октябрь 2026г***.
- Для сбора объявлений использовался только один источник - сайт cian.ru
- С течением времени оригинальные объявления снимаются с публикации и перестают быть доступны для повторного просмотра.
- Собранная информация не является генеральной совокупностью и не применима для эффективной оценки рынка недвижимости, но имеет достаточную ценность в качестве учебного материала.
### Очистка и обработка данных
При запуске программы все содержимое БД объединяется в один pandas.DataFrame-объект с условием "один объект сделки -> одно объявление -> последняя известная цена".  
Для удобства исследований производятся следующие очистки:  
- **Фильтрация по валюте сделки** - в подавляющем большинстве случаев указанная валюта сделки - "Рубли РФ". Объявления с иной валютой сделки исключаются из итоговой таблицы.
- **Фильтрация по типу сделки** - подавляющее большинство объявлений имеют тип **"Продажа"**. Объявления с иным типом сделки исключаются из итоговой таблицы.
- **Фильтрация по указанию цены** - объявления без указания цены сделки не имеют ценности для исследований и исключаются из итоговой таблицы.
- **Фильтрация по типу недвижимости** - подавляющее большинство объектов недвижимости имеют тип **квартира**. Объекты иного типа недвижимости исключаются из итоговой таблицы.
- **Избавление от дубликатов объявлений** - основным идентификатором объявлений в итоговой таблице принято считать поле **cian_listing_id**. При фильтрации из групп повторяющихся объявлений в итоговой таблице сохраняются единственные экземпляры с наиболее поздней датой обновления цены.
- **Фильтрация по экстремально-высоким изменениям цен** - данный пункт необходим для исследования истории изменения цен. При извлечении данных из таблицы **price_history** к итоговой табице добавляется поле статуса изменения цены, которое имеет пять "уровней подозрительности": **single_record**, **valid**, **low**, **high**, **extreme**. Как правило, объявления со статусом **extreme** имеют в истории резкие изменения цены в десятки, сотни и тысячи раз, что характерно для *"ошибки в количестве нулей"*. Количество таких объявлений незначительное и они исключаются из итоговой таблицы для уменьшения искажений при визуализации и обработке данных.  
##### Методика определения уровня подозрительности изменения цен
- Для объявлений, имеющих единственную запись цены устанавливается уровень **single_record**, так как изменения цены отследить невозможно.
- Для объявлений с изменявшейся ценой рассчитывается отношение абсолютной величины изменения к предыдущей цене (далее *diff_grade*).
- Для всего массива значений *diff_grade* рассчитывается верхняя граница ожидаемых значений по формуле:  
  *high = 75th_quantile + 1.5 x (75th_quantile - 25th_quantile)*
- Строки таблицы **price_histiry**, имеющие значение *diff_grade* ниже *high* получают уровень подозрительности **valid**.
- Остальные строки передаются в следующую аналогичную итерацию, для определения уровня **low**.
- Уровни **high** и **extreme** определяются по такому же принципу с последовательным сокращением выборки и ростом значения *high*.
##### Информация о количестве исключенных из итоговой таблицы строк логируется
##### Ниже преведены некоторые сведения о состоянии БД на 9.11.2026 и о результатах предварительной фильтрации

| Метрика | Состояние |
|:---|---:|
| Всего строк в таблице **apartments** | 7506 |
| Всего строк в таблице **listings** | 7506 |
| Всего строк в таблице **price_history** | 40317 |
| Невалидных объявлений по валюте сделки | 1 |
| Всего строк, не явлющихся объявлениями о продаже | 164 |
| Объявлений без указания цены сделки | 164 |
| Объектов недвижимости, не являющихся квартирами | 295 |
| Общее количество повторных записей объявлений | 980 |
| **single_record** | 2061 |
| **valid** | 3689 |
| **low** | 333 |
| **high** | 10 |
| **extreme** (исключенные объявления) | 25 |
| Количество уникальных объявлений для дальнейшей обработки | 6093 |
**Статусы valid, low, high в итоговой таблице соответствуют последней записи изменения цены*


In [1]:
# Импорты и извлечение данных из БД
import sys
sys.path.append("..")

import os

from data_base.apartments_data_cleaning import clean_listings_data, get_original_listings
from dotenv import load_dotenv
from IPython.display import display, Markdown
from reports.reports_backend import show_markdown_missing, show_cbc_profit

load_dotenv()

DB_NAME = os.getenv("DB_NAME", "apartments.db")
PATH_TO_DB = f"../{DB_NAME}"
original_data = get_original_listings(PATH_TO_DB, "apartments", "listings", "price_history")
cleaned_data = clean_listings_data(original_data)

INFO:root:Для признака cian_building_code подтверждена уникальность собственных географических признаков.
        Часть географических признаков будет автоматически восстановлена с помощью cian_building_code
INFO:root:Восстановлено 1 пропусков признака district на основании уникальности признака cian_building_code


In [2]:
# Статистика пропущенных значений
show_markdown_missing(cleaned_data)

## Статистика пропущенных значений
| Признак | Всего пропусков | Процент пропусков |
|:---|:---:|:---:|
| street | 2349 | 38.55 % |
| district | 2010 | 32.99 % |
| cian_building_code | 1972 | 32.37 % |
| building_number | 1972 | 32.37 % |
| year_built | 1489 | 24.44 % |
| city | 822 | 13.49 % |
| ceiling_height | 706 | 11.59 % |
| cian_builder_id | 604 | 9.91 % |
| builder_name | 604 | 9.91 % |
| building_material | 587 | 9.63 % |

## Признак **cian_building_code**
### Однозначный идентификатор
В ходе исследований обнаружено, что **cian_building_code** однозначно идентифицирует географическое расположение объекта объявления.
Исходя из этого свойства, принято решение использовать **cian_building_code** для восстановления пропусков других географических признаков: **city**, **district**, **street**, **building_number**.  
Во время извлечения данных из БД проверяется уникальность **cian_building_code**. Результаты проверки логируются с уровнем **INFO**, если проверка подтверждает уникальность. Если уникальность не подтверждается, выводится сообщение с уровнем **ERROR** и требованием провести диагностику данных.

In [3]:
# Проверка уникальности cian_building_code в соответствии с другими географическими признаками
GEOGRAPHIC_SIGNS = ["city", "district", "street", "building_number"]
unique_success = True
message = ""
for sign in GEOGRAPHIC_SIGNS:
    filled_data = original_data[[sign, "cian_building_code"]].dropna().copy()
    unique_pair = filled_data.groupby("cian_building_code").nunique().nunique().size
    if unique_pair > 1:
        if unique_success:
            message += "### Уникальность признака cian_building_code не подтверждена!\n" 
            unique_success = False
        message += f"Обнаружены неоднозначные значения признака **{sign}**  \n"
if unique_success:
        message = "Уникальность признака cian_building_code подтверждена"
display(Markdown(message))

Уникальность признака cian_building_code подтверждена

### Потенциальная польза **cian_building_code** для восстановления географических признаков
Несмотря на способность признака **cian_building_code** в некоторых случаях однозначно определить местоположение объектов объявлений, в собранных к 9.11.2026 данных, обнаружена закономерность пропусков (**MNAR**) данного признака, не позволяющая эффективно восстанавливать данные. Объявления условно можно разделить на три категории:
- все строки с одинаковым **cian_building_code** имеют один и тот же указанный географический признак
- все строки с одинаковым **cian_building_code** имеют пропуск географического признака
- строки с одинаковым **cian_building_code** имеют, и пропуски географического признака, и заполненные значения

Первые две категории не имеют *пересечений данных* и бесполезны для восстановления географических признаков. Подавляющее большинство строк в таблице относится к этим двум категориям. Основные предположительные причины возникновения таких особенностей: систематические пропуски в источнике данных **cian.ru**, ошибки при сборе данных, неоднозначная структура объявлений в источнике.  
Ниже приведена таблица, позволяющая оценить пользу **cian_building_code** для восстановления географических признаков на 9.11.2026  
| Признак | Указанных значений | Пропусков | Можно восстановить |
| :--- | :--- | :--- | :---: |
| city | 4424 | 596 | 0 |
| district | 3163 | 1857 | 1 |
| street | 4312 | 708 | 0 |
| building_number | 5020 | 0 | 0 |

In [4]:
show_cbc_profit(original_data)

### Польза cian_building_code при восстановлении географических признаков на 2026-11-15
| Признак | Указанных значений | Пропусков | Можно восстановить |
| :--- | :---: | :---: | :---: |
| city | 4424 | 596 | 0 |
| district | 3163 | 1857 | 1 |
| street | 4312 | 708 | 0 |
| building_number | 5020 | 0 | 0 |

*Для расчета показателей используются строки с указанными значениями **cian_building_code***